## 1. ライブラリのインポートと設定

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from pathlib import Path
import json
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score, average_precision_score
import lightgbm as lgb
from scipy import sparse

# 日本語フォント設定（必要に応じて）
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# スタイル設定
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

### 日本語フォントの準備

In [ ]:
# 日本語フォントを明示的に設定
plt.rcParams['font.sans-serif'] = ['Hiragino Sans', 'YuGothic', 'IPAexGothic', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の文字化け対策

# スタイル設定
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("使用中のフォント:", plt.rcParams['font.sans-serif'][0])

In [ ]:
# フォントキャッシュをクリアして再構築
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# フォントキャッシュを削除
fm._load_fontmanager(try_read_cache=False)

# 日本語フォントを設定
plt.rcParams['font.sans-serif'] = ['Hiragino Sans', 'YuGothic', 'IPAexGothic']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

print("フォント設定完了")
print("現在のfont.sans-serif:", plt.rcParams['font.sans-serif'])
print("現在のfont.family:", plt.rcParams['font.family'])

## 2. データの読み込み

分析結果ディレクトリを指定して、必要なファイルを読み込みます。

In [ ]:
# 分析結果ディレクトリを指定(最新のものを使用する場合は適宜変更)
result_dir = Path("results/experiments/feature_selection_prompt_last_token_20251129_214443")

# データディレクトリ
data_dir = result_dir / "data"

# メインデータの読み込み
feature_metrics = pd.read_csv(data_dir / "feature_metrics_full.csv")
candidates_suppress = pd.read_csv(data_dir / "candidates_suppress.csv")
candidates_amplify = pd.read_csv(data_dir / "candidates_amplify.csv")

# Candidate Type列を追加
suppress_ids = set(candidates_suppress['Feature_ID'].values)
amplify_ids = set(candidates_amplify['Feature_ID'].values)

def assign_candidate_type(feature_id):
    if feature_id in suppress_ids:
        return 'Suppress'
    elif feature_id in amplify_ids:
        return 'Amplify'
    else:
        return 'Other'

feature_metrics['Candidate Type'] = feature_metrics['Feature_ID'].apply(assign_candidate_type)

print(f"全特徴数: {len(feature_metrics)}")
print(f"抑制候補数: {len(candidates_suppress)}")
print(f"増幅候補数: {len(candidates_amplify)}")
print(f"\nCandidate Type分布:")
print(feature_metrics['Candidate Type'].value_counts())
print("\n特徴メトリクスのカラム:")
print(feature_metrics.columns.tolist())

### データの概要確認

In [ ]:
feature_metrics

In [ ]:
# 基本統計量
print("=== 基本統計量 ===")
feature_metrics[['Freq Diff Base-Syc', 'Log Ratio Syc/Base', 'Diff Base-Syc', 'SHAP Correlation', 'Suppression Score', 'Amplification Score']].describe()

In [ ]:
# 上位抑制候補
print("\n=== 上位抑制候補（Top 5） ===")
candidates_suppress.head()

In [ ]:
# 上位増幅候補
print("\n=== 上位増幅候補（Top 5） ===")
candidates_amplify[["Feature_ID","Specificity", "Freq NonSyc (%)", "Mean Intensity Base", "Mean Intensity Syc", "Suppression Score", "Amplification Score"]].head()

### 足りない指標を追加

In [ ]:
def calc_freq_dff_base_nonsyc(row):
    return row['Freq NonSyc (%)'] - row['Freq Base (%)']

In [ ]:
# 各データに対して計算を適用
feature_metrics['Freq Diff Base-NonSyc'] = feature_metrics.apply(calc_freq_dff_base_nonsyc, axis=1)
candidates_suppress['Freq Diff Base-NonSyc'] = candidates_suppress.apply(calc_freq_dff_base_nonsyc, axis=1)
candidates_amplify['Freq Diff Base-NonSyc'] = candidates_amplify.apply(calc_freq_dff_base_nonsyc, axis=1)

## 3. 介入候補の可視化

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from matplotlib.patches import Patch

# プロット準備: 介入候補のヒートマップ（指標ごとに正規化）

# 1. データ抽出
top_suppress = candidates_suppress.head(20).copy()
top_amplify = candidates_amplify.head(20).copy()

# 抑制・増幅候補を結合
top_suppress['Candidate'] = 'Suppress'
top_amplify['Candidate'] = 'Amplify'
top_candidates = pd.concat([top_suppress, top_amplify], ignore_index=True)

# Feature_IDをインデックスに設定
top_candidates = top_candidates.set_index('Feature_ID')

# 2. ヒートマップ用の指標を選択
# 頻度系、強度系、スコア系の主要指標を選択
heatmap_columns = [
    'Freq Syc (%)', 'Freq NonSyc (%)', 'Freq Base (%)', 
    'Specificity', 'Consistency',
    'Mean Intensity Syc', 'Mean Intensity NonSyc', 'Mean Intensity Base',
    'Diff Base-Syc', 'Log Ratio Syc/Base',
    'SHAP Correlation', 'Suppression Score', 'Amplification Score'
]

# 選択した指標のみを抽出
heatmap_data = top_candidates[heatmap_columns].copy()

# 3. 列ごとにMin-Max正規化（0〜1にスケーリング）

scaler = MinMaxScaler()
heatmap_data_normalized = pd.DataFrame(
    scaler.fit_transform(heatmap_data),
    index=heatmap_data.index,
    columns=heatmap_data.columns
)

# 4. ヒートマップの作成
fig, ax = plt.subplots(figsize=(14, 10))

# Seabornヒートマップ（正規化後のデータ、注釈なし）
sns.heatmap(
    heatmap_data_normalized,
    cmap='Blues',
    annot=False,  # 正規化後の値は表示しない
    fmt='.2f',
    linewidths=0.5,
    linecolor='lightgray',
    cbar_kws={'label': 'Normalized Value (0-1)'},
    ax=ax
)

# 抑制候補と増幅候補の境界線を引く（10件目と11件目の間）
ax.axhline(y=20, color='red', linewidth=3, linestyle='--', label='Suppress/Amplify Border')

# Y軸ラベル（特徴ID）を見やすく
ax.set_ylabel('Feature ID', fontsize=12, fontweight='bold')
ax.set_xlabel('Metrics (Normalized)', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Suppress & Amplify Candidates Heatmap\n(Each Column Normalized 0-1)', 
             fontsize=14, fontweight='bold', pad=20)

# Y軸ラベルを回転させて読みやすく
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)

# 凡例追加（境界線の説明）
legend_elements = [
    Patch(facecolor='white', edgecolor='red', linestyle='--', linewidth=2, label='Suppress ↑ / Amplify ↓')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

# # 保存
# fig.savefig(result_dir / "figures" / "heatmap_top_candidates_normalized.png", dpi=300, bbox_inches='tight')
# print(f"図を保存しました: {result_dir / 'figures' / 'heatmap_top_candidates_normalized.png'}")

In [ ]:
# プロット: SpecificityとConsistencyの散布図
# X軸: Specificity, Y軸: Consistency, サイズ: Log Ratio Syc/Base

fig, ax = plt.subplots(figsize=(16, 10))

# サイズの正規化（Log Ratio Syc/Baseを使用）
size_column = 'Log Ratio Syc/Base'
# 絶対値をとってサイズに変換（最小10、最大500）
min_size, max_size = 10, 500
log_ratio_abs = feature_metrics[size_column].abs()
size_scale = (log_ratio_abs - log_ratio_abs.min()) / (log_ratio_abs.max() - log_ratio_abs.min())
feature_metrics['bubble_size_specificity'] = min_size + size_scale * (max_size - min_size)

# Candidate Typeごとにデータを分割
df_other = feature_metrics[feature_metrics['Candidate Type'] == 'Other']
df_suppress = feature_metrics[feature_metrics['Candidate Type'] == 'Suppress']
df_amplify = feature_metrics[feature_metrics['Candidate Type'] == 'Amplify']

# その他の特徴（背景）
ax.scatter(
    df_other['Specificity'],
    df_other['Consistency'],
    s=df_other['bubble_size_specificity'],
    c='lightgray',
    alpha=0.3,
    marker='.',
    label='Other',
    zorder=1
)

# 増幅候補（青）
ax.scatter(
    df_amplify['Specificity'],
    df_amplify['Consistency'],
    s=df_amplify['bubble_size_specificity'],
    c='blue',
    alpha=0.7,
    marker='o',
    edgecolors='navy',
    linewidths=1.5,
    label='Amplify Candidates',
    zorder=3
)

# 抑制候補（赤）
ax.scatter(
    df_suppress['Specificity'],
    df_suppress['Consistency'],
    s=df_suppress['bubble_size_specificity'],
    c='red',
    alpha=0.7,
    marker='o',
    edgecolors='darkred',
    linewidths=1.5,
    label='Suppress Candidates',
    zorder=3
)

# 上位5つの抑制候補に注釈を追加
for idx, row in top_suppress.iterrows():
    feature_id = row['Feature_ID']
    feature_row = feature_metrics[feature_metrics['Feature_ID'] == feature_id].iloc[0]
    ax.annotate(
        f"ID:{feature_id}",
        xy=(feature_row['Specificity'], feature_row['Consistency']),
        xytext=(10, 10),
        textcoords='offset points',
        fontsize=9,
        color='darkred',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='pink', alpha=0.7),
        arrowprops=dict(arrowstyle='->', color='red', lw=1)
    )

# 上位5つの増幅候補に注釈を追加
for idx, row in top_amplify.iterrows():
    feature_id = row['Feature_ID']
    feature_row = feature_metrics[feature_metrics['Feature_ID'] == feature_id].iloc[0]
    ax.annotate(
        f"ID:{feature_id}",
        xy=(feature_row['Specificity'], feature_row['Consistency']),
        xytext=(10, -15),
        textcoords='offset points',
        fontsize=9,
        color='darkblue',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7),
        arrowprops=dict(arrowstyle='->', color='blue', lw=1)
    )

# 軸ラベルとタイトル
ax.set_xlabel('Specificity (迎合時特異性)', fontsize=13, fontweight='bold')
ax.set_ylabel('Consistency (迎合時一貫性)', fontsize=13, fontweight='bold')
ax.set_title('SpecificityとConsistencyの散布図\n(バブルサイズ: Log Ratio Syc/Base)', 
             fontsize=15, fontweight='bold', pad=20)

# 凡例
ax.legend(loc='upper left', fontsize=11, framealpha=0.9)

# グリッド
ax.grid(True, alpha=0.3)

# 参照線（高Specificity、高Consistencyの領域を示す）
ax.axhline(y=0.7, color='green', linestyle=':', alpha=0.5, linewidth=1.5)
ax.axvline(x=0.7, color='green', linestyle=':', alpha=0.5, linewidth=1.5)

# 領域の説明テキスト
# ax.text(0.98, 0.98, '← 高Specificity &\n高Consistency領域', 
#         transform=ax.transAxes, fontsize=11, 
#         verticalalignment='top', horizontalalignment='right',
#         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.6))

plt.tight_layout()
plt.show()

# 図を保存
# fig.savefig(result_dir / "figures" / "plot_specificity_consistency.png", dpi=300, bbox_inches='tight')
# print(f"図を保存しました: {result_dir / 'figures' / 'plot_specificity_consistency.png'}")

### プロットA: 強度と頻度の総合分析 (Bubble Chart)

In [ ]:
# プロットA: 強度と頻度の総合分析 (Bubble Chart)
# X軸: Log Ratio Syc/Base, Y軸: Diff Base-Syc, サイズ: Freq Diff Base-Syc

fig, ax = plt.subplots(figsize=(16, 10))

# サイズの正規化（負の値も考慮）
size_column = 'Freq Diff Base-Syc'
# 絶対値をとってサイズに変換（最小10、最大500）
min_size, max_size = 10, 500
freq_diff_abs = feature_metrics[size_column].abs()
size_scale = (freq_diff_abs - freq_diff_abs.min()) / (freq_diff_abs.max() - freq_diff_abs.min())
feature_metrics['bubble_size'] = min_size + size_scale * (max_size - min_size)

# Candidate Typeごとにデータを分割
df_other = feature_metrics[feature_metrics['Candidate Type'] == 'Other']
df_suppress = feature_metrics[feature_metrics['Candidate Type'] == 'Suppress']
df_amplify = feature_metrics[feature_metrics['Candidate Type'] == 'Amplify']

# その他の特徴（背景）
ax.scatter(
    df_other['Log Ratio Syc/Base'],
    df_other['Diff Base-Syc'],
    s=df_other['bubble_size'],
    c='lightgray',
    alpha=0.3,
    marker='.',
    label='Other',
    zorder=1
)

# 増幅候補（青）
ax.scatter(
    df_amplify['Log Ratio Syc/Base'],
    df_amplify['Diff Base-Syc'],
    s=df_amplify['bubble_size'],
    c='blue',
    alpha=0.7,
    marker='*',
    edgecolors='navy',
    linewidths=1.5,
    label='Amplify Candidates',
    zorder=3
)

# 抑制候補（赤）
ax.scatter(
    df_suppress['Log Ratio Syc/Base'],
    df_suppress['Diff Base-Syc'],
    s=df_suppress['bubble_size'],
    c='red',
    alpha=0.7,
    marker='X',
    edgecolors='darkred',
    linewidths=1.5,
    label='Suppress Candidates',
    zorder=3
)

# 上位5つの候補に注釈を追加
top_suppress = candidates_suppress.head(5)
top_amplify = candidates_amplify.head(5)

for idx, row in top_suppress.iterrows():
    feature_id = row['Feature_ID']
    feature_row = feature_metrics[feature_metrics['Feature_ID'] == feature_id].iloc[0]
    ax.annotate(
        f"ID:{feature_id}",
        xy=(feature_row['Log Ratio Syc/Base'], feature_row['Diff Base-Syc']),
        xytext=(10, 10),
        textcoords='offset points',
        fontsize=9,
        color='darkred',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='pink', alpha=0.7),
        arrowprops=dict(arrowstyle='->', color='red', lw=1)
    )

for idx, row in top_amplify.iterrows():
    feature_id = row['Feature_ID']
    feature_row = feature_metrics[feature_metrics['Feature_ID'] == feature_id].iloc[0]
    ax.annotate(
        f"ID:{feature_id}",
        xy=(feature_row['Log Ratio Syc/Base'], feature_row['Diff Base-Syc']),
        xytext=(10, -15),
        textcoords='offset points',
        fontsize=9,
        color='darkblue',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7),
        arrowprops=dict(arrowstyle='->', color='blue', lw=1)
    )

# 軸ラベルとタイトル
ax.set_xlabel('Log Ratio Syc/Base (対数倍率)', fontsize=13, fontweight='bold')
ax.set_ylabel('Diff Base-Syc (強度差分)', fontsize=13, fontweight='bold')
ax.set_title('プロットA: 強度と頻度の総合分析 (Bubble Chart)', fontsize=15, fontweight='bold', pad=20)

# 凡例
ax.legend(loc='upper left', fontsize=11, framealpha=0.9)

# グリッドと参照線
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.6, linewidth=1.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.6, linewidth=1.5)

plt.tight_layout()
plt.show()

# 図を保存
# fig.savefig(result_dir / "figures" / "plot_a_intensity_frequency_bubble.png", dpi=300, bbox_inches='tight')
# print(f"図を保存しました: {result_dir / 'figures' / 'plot_a_intensity_frequency_bubble.png'}")

In [ ]:
# プロットB: 活性化頻度の変化 (Frequency Shift)
# X軸: Freq Base (%), Y軸: Freq Syc (%)

fig, ax = plt.subplots(figsize=(14, 12))

# その他の特徴（背景）
ax.scatter(
    df_other['Freq Base (%)'],
    df_other['Freq Syc (%)'],
    s=20,
    c='lightgray',
    alpha=0.3,
    marker='.',
    label='Other',
    zorder=1
)

# 増幅候補（青）
ax.scatter(
    df_amplify['Freq Base (%)'],
    df_amplify['Freq Syc (%)'],
    s=200,
    c='blue',
    alpha=0.7,
    marker='*',
    edgecolors='navy',
    linewidths=1.5,
    label='Amplify Candidates',
    zorder=3
)

# 抑制候補（赤）
ax.scatter(
    df_suppress['Freq Base (%)'],
    df_suppress['Freq Syc (%)'],
    s=200,
    c='red',
    alpha=0.7,
    marker='X',
    edgecolors='darkred',
    linewidths=1.5,
    label='Suppress Candidates',
    zorder=3
)

# y=x の補助線
max_freq = max(feature_metrics['Freq Base (%)'].max(), feature_metrics['Freq Syc (%)'].max())
ax.plot([0, max_freq], [0, max_freq], 'k--', alpha=0.5, linewidth=2, label='y=x (no change)')

# 上位5つの候補に注釈を追加
for idx, row in top_suppress.iterrows():
    feature_id = row['Feature_ID']
    feature_row = feature_metrics[feature_metrics['Feature_ID'] == feature_id].iloc[0]
    ax.annotate(
        f"ID:{feature_id}",
        xy=(feature_row['Freq Base (%)'], feature_row['Freq Syc (%)']),
        xytext=(10, 10),
        textcoords='offset points',
        fontsize=9,
        color='darkred',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='pink', alpha=0.7),
        arrowprops=dict(arrowstyle='->', color='red', lw=1)
    )

for idx, row in top_amplify.iterrows():
    feature_id = row['Feature_ID']
    feature_row = feature_metrics[feature_metrics['Feature_ID'] == feature_id].iloc[0]
    ax.annotate(
        f"ID:{feature_id}",
        xy=(feature_row['Freq Base (%)'], feature_row['Freq Syc (%)']),
        xytext=(-60, -15),
        textcoords='offset points',
        fontsize=9,
        color='darkblue',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7),
        arrowprops=dict(arrowstyle='->', color='blue', lw=1)
    )

# 軸ラベルとタイトル
ax.set_xlabel('Freq Base (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Freq Syc (%)', fontsize=13, fontweight='bold')
ax.set_title('プロットB: 活性化頻度の変化 (Frequency Shift)', fontsize=15, fontweight='bold', pad=20)

# 凡例
ax.legend(loc='upper left', fontsize=11, framealpha=0.9)

# グリッド
ax.grid(True, alpha=0.3)

# 領域の説明
ax.text(0.98, 0.02, '↑ Syc時に頻度増加', 
        transform=ax.transAxes, fontsize=11, 
        verticalalignment='bottom', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

ax.text(0.02, 0.9, '← Base時に頻度増加', 
        transform=ax.transAxes, fontsize=11,
        verticalalignment='top', horizontalalignment='left',
        bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.7))

plt.tight_layout()
plt.show()

# 図を保存
# fig.savefig(result_dir / "figures" / "plot_b_frequency_shift.png", dpi=300, bbox_inches='tight')
# print(f"図を保存しました: {result_dir / 'figures' / 'plot_b_frequency_shift.png'}")

## 4. モデル性能の確認

summary.txtに記録されたモデル性能を可視化します。

In [ ]:
# summary.txtからモデル性能の数値を読み込む
summary_path = result_dir / "summary.txt"

# モデル性能を抽出
model_metrics = {}
with open(summary_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if 'ROC AUC:' in line:
            model_metrics['ROC AUC'] = float(line.split(':')[1].strip())
        elif 'Average Precision:' in line:
            model_metrics['Average Precision'] = float(line.split(':')[1].strip())
        elif 'Accuracy:' in line:
            model_metrics['Accuracy'] = float(line.split(':')[1].strip())
        elif 'Precision:' in line:
            model_metrics['Precision'] = float(line.split(':')[1].strip())
        elif 'Recall:' in line:
            model_metrics['Recall'] = float(line.split(':')[1].strip())
        elif 'F1 Score:' in line:
            model_metrics['F1 Score'] = float(line.split(':')[1].strip())
        elif '混同行列:' in line:
            # 次の2行から混同行列を取得
            tn_fp_line = lines[i+1].strip()
            fn_tp_line = lines[i+2].strip()
            # TN: 2691, FP: 1148 のような形式
            tn = int(tn_fp_line.split('TN:')[1].split(',')[0].strip())
            fp = int(tn_fp_line.split('FP:')[1].strip())
            fn = int(fn_tp_line.split('FN:')[1].split(',')[0].strip())
            tp = int(fn_tp_line.split('TP:')[1].strip())
            model_metrics['confusion_matrix'] = [[tn, fp], [fn, tp]]

print("=== モデル性能 ===")
for key, value in model_metrics.items():
    if key != 'confusion_matrix':
        print(f"{key}: {value:.4f}")
print(f"\n混同行列:")
print(f"  TN: {model_metrics['confusion_matrix'][0][0]}, FP: {model_metrics['confusion_matrix'][0][1]}")
print(f"  FN: {model_metrics['confusion_matrix'][1][0]}, TP: {model_metrics['confusion_matrix'][1][1]}")

### モデル性能の可視化

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 3つのプロットを作成
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 混同行列
cm = np.array(model_metrics['confusion_matrix'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Syc', 'Syc'])
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('混同行列', fontsize=14, fontweight='bold')
axes[0].grid(False)

# 2. 主要メトリクスのバープロット
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC', 'Average Precision']
values = [model_metrics[m] for m in metrics_to_plot]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

bars = axes[1].barh(metrics_to_plot, values, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Score', fontsize=12, fontweight='bold')
axes[1].set_title('モデル性能メトリクス', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 1)
axes[1].grid(axis='x', alpha=0.3)

# 数値をバーに表示
for i, (bar, value) in enumerate(zip(bars, values)):
    axes[1].text(value + 0.02, i, f'{value:.3f}', va='center', fontsize=10, fontweight='bold')

# 3. ROC曲線とPR曲線の情報表示（実際の曲線は描けないため、AUC値のみ表示）
info_text = f"""
モデル性能サマリー
━━━━━━━━━━━━━━━━━
ROC AUC: {model_metrics['ROC AUC']:.4f}
Average Precision: {model_metrics['Average Precision']:.4f}

━━━━━━━━━━━━━━━━━
分類性能:
━━━━━━━━━━━━━━━━━
Accuracy:  {model_metrics['Accuracy']:.4f}
Precision: {model_metrics['Precision']:.4f}
Recall:    {model_metrics['Recall']:.4f}
F1 Score:  {model_metrics['F1 Score']:.4f}

━━━━━━━━━━━━━━━━━
混同行列:
━━━━━━━━━━━━━━━━━
TN: {cm[0,0]:>4d}  FP: {cm[0,1]:>4d}
FN: {cm[1,0]:>4d}  TP: {cm[1,1]:>4d}

━━━━━━━━━━━━━━━━━
注: ROC曲線とPR曲線を描画
するには、テストデータの
予測確率が必要です。
"""

axes[2].text(0.1, 0.5, info_text, fontsize=11, family='monospace',
             verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[2].axis('off')
axes[2].set_title('性能サマリー', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# 保存
# fig.savefig(result_dir / "figures" / "model_performance_summary.png", dpi=300, bbox_inches='tight')
# print(f"図を保存しました: {result_dir / 'figures' / 'model_performance_summary.png'}")

## 5. スコア分布の確認

In [ ]:
def show_feature_details(feature_id, df):
    """
    指定された特徴IDの詳細情報を表示
    
    Parameters:
    -----------
    feature_id : int
        特徴ID
    df : pd.DataFrame
        特徴メトリクスのデータフレーム
    """
    feature = df[df['Feature_ID'] == feature_id]
    
    if len(feature) == 0:
        print(f"特徴ID {feature_id} は見つかりませんでした。")
        return
    
    feature = feature.iloc[0]
    
    print("=" * 60)
    print(f"特徴ID: {feature_id}")
    print("=" * 60)
    print(f"Candidate Type: {feature['Candidate Type']}")
    print("\n--- 基本指標 ---")
    print(f"Freq Base (%): {feature['Freq Base (%)']:.2f}")
    print(f"Freq Syc (%): {feature['Freq Syc (%)']:.2f}")
    print(f"Freq NonSyc (%): {feature['Freq NonSyc (%)']:.2f}")
    print(f"Mean Intensity Base: {feature['Mean Intensity Base']:.6f}")
    print(f"Mean Intensity Syc: {feature['Mean Intensity Syc']:.6f}")
    print(f"Mean Intensity NonSyc: {feature['Mean Intensity NonSyc']:.6f}")
    print("\n--- 重要な指標 ---")
    print(f"Log Ratio Syc/Base: {feature['Log Ratio Syc/Base']:.6f}")
    print(f"Diff Base-Syc: {feature['Diff Base-Syc']:.6f}")
    print(f"Freq Diff Base-Syc: {feature['Freq Diff Base-Syc']:.2f}")
    print(f"Specificity: {feature['Specificity']:.6f}")
    print(f"Suppression Score: {feature['Suppression Score']:.6f}")
    print(f"Amplification Score: {feature['Amplification Score']:.6f}")
    print("=" * 60)

# 使用例: 上位候補を表示
if len(candidates_suppress) > 0:
    top_suppress_id = candidates_suppress.iloc[0]['Feature_ID']
    print("【抑制候補 Top 1】")
    show_feature_details(top_suppress_id, feature_metrics)

if len(candidates_amplify) > 0:
    top_amplify_id = candidates_amplify.iloc[0]['Feature_ID']
    print("\n【増幅候補 Top 1】")
    show_feature_details(top_amplify_id, feature_metrics)

In [ ]:
# 任意の特徴IDを指定して詳細を確認
# 以下の数値を変更して実行してください
custom_feature_id = 2705  # 確認したい特徴IDを指定
show_feature_details(custom_feature_id, feature_metrics)

## 7. まとめ

このノートブックでは以下を実行しました:

1. **データ読み込みと前処理**: 特徴メトリクスと候補リストを読み込み、Candidate Type列を追加
2. **プロットA - 強度と頻度の総合分析**: Log Ratio vs Diff Base-Sycのバブルチャートで、Freq Diff Base-Sycをサイズに反映。候補特徴を色とマーカーで強調し、上位5つに注釈を表示
3. **プロットB - 活性化頻度の変化**: Freq Base vs Freq Sycの散布図で、y=x補助線により頻度変化の方向性を可視化
4. **モデル性能の確認**: summary.txtから性能メトリクスを読み込み、混同行列とバープロットで表示
5. **スコア分布の確認**: Suppression ScoreとAmplification Scoreのヒストグラムで、95パーセンタイルを表示
6. **特徴詳細の確認**: 任意の特徴IDの詳細情報を表示する関数を提供

### 次のステップ
- 特定された介入候補特徴（Suppress/Amplify）を用いて、ステップ4の介入実験を実施
- 介入効果の評価と分析を行い、迎合性抑制の効果を検証